# VCM — Final research review and complete command ledger

## TL;DR

**Retain `phone_s53` as the research reference. There is no qualified deployment winner.** One causal 64-channel CNN; **113,064 parameters, 458,230-byte FP32 ONNX, 32,064-byte neural cache**. Outputs are 39 phonemes plus CTC blank: constrained phonetic recognition, not unrestricted transcription.

The V6 epoch-zero parent beats its adapted checkpoints on the matched development set. Incompatible test catalogs cannot rank models. V5's verifier is not calibrated for V6's expanded catalog. V6 refuses every final command; raw success is not accepted success or an executed action.

The ledger retains **36,972 rows from 79 reports**, including failed alternatives and train/dev/calibration evaluations. All included acoustic rows are synthetic; repeated evaluations are not independent recordings. Most historical WAVs are missing. Independent human speech, Pi4, full ten-use-case and acceptance qualification remain incomplete. No acoustic weights were trained in this final-suite pass.

This compact Git notebook reads the restored payload. The accompanying executed download embeds all evidence and complete tables.

## 1. Architecture → datasets → typed result

```mermaid
flowchart LR
 A[16 kHz mono PCM] --> B[400 Hann / 160 hop / 512 FFT]
 B --> C[40 log-mel bands / fixed normalization]
 C --> D[Conv 40 to 64 / kernel 3]
 D --> E[Five causal Conv blocks / kernel 5 / dilation 1,2,4,8,16]
 E --> F[Linear projection 64 to 40]
 F --> G[39 phonemes plus blank]
 H[HA aliases and dummy music title-artist lists] --> I[Pronunciations and stable IDs]
 I --> J[Shared-prefix complete-command CTC]
 G --> J
 J --> K[Provisional intent and complete parameters]
 K --> L[Endpoint / capture integrity / model and catalog identity]
 L --> M[Calibrated acceptance or clarification]
 M --> N[Diagnostic only / no actions]
```

```mermaid
flowchart TD
 A[Own synthetic training] --> B[Retained phone_s53]
 B --> C[V6 adaptation / development comparison]
 C --> D[Epoch-zero parent retained]
 E[Archived V1-V6 manifests and predictions] --> F[Complete regression ledger]
 G[44 new synthetic scope WAVs] --> F
 F --> H[Release gate]
 I[Human and Pi4 evidence missing] -. required .-> H
 D --> J[FP32 ONNX reference]
 J --> K[Exact float32 browser recipe]
 K --> L[WASM CNN and CTC]
 L --> M[Real tensor probes / WebGL or Canvas 3D]
```

Inspection speed changes scheduling, never PCM. A provisional 70 may become 73. The viewer is separate from the Pi inference process.

In [ ]:
import gzip, json, html
from pathlib import Path
from collections import defaultdict
from IPython.display import HTML, display

payload = Path('final_suite/data/notebook_evidence.json.gz')
if not payload.is_file():
    raise FileNotFoundError('Restore VCM_FINAL_PAYLOAD.zip; see FINAL_REVIEW.md')
evidence = json.loads(gzip.decompress(payload.read_bytes()))
rows = evidence['rows']

def show_table(values, columns):
    def text(value):
        return json.dumps(value, ensure_ascii=False, sort_keys=True) if isinstance(value, (dict, list)) else str(value)
    body = '<table border="1" style="font-size:12px;border-collapse:collapse"><tr>'
    body += ''.join('<th>' + html.escape(c) + '</th>' for c in columns) + '</tr>'
    for row in values:
        body += '<tr>' + ''.join('<td>' + html.escape(text(row.get(c, ''))) + '</td>' for c in columns) + '</tr>'
    return body + '</table>'

def show_complete(title, values):
    groups = defaultdict(list)
    for row in values:
        groups[row['trial'] + ' / ' + row['dataset']].append(row)
    columns = ['trial', 'dataset', 'partition', 'voice', 'id', 'verbatim', 'expected', 'raw_output', 'recorded_status', 'accepted', 'outcome']
    parts = ['<h3>' + html.escape(title) + ' — ' + str(len(values)) + ' rows</h3>']
    for name, group in groups.items():
        parts.append('<details><summary>' + html.escape(name) + ' (' + str(len(group)) + ' rows)</summary>' + show_table(group, columns) + '</details>')
    display(HTML(''.join(parts)))

print(len(rows), 'evaluation rows; not independent test recordings.')

## 2. Exact model and frontend

Periodic Hann window 400, hop 160, FFT 512, 40 HTK mel bins **20–7,600 Hz**, log floor 1e-8, saved fixed normalization. All layers have stride one, bias and causal history. ReLU except the final projection; receptive field 127 frames. No pooling, Transformer or second acoustic model.

The browser validates compatible recipe/weights/catalog bundles, not arbitrary ONNX graphs. Optional ONNX references are hash checked. WASM computes actual tensors; WebGL draws them and is not WebGPU inference. Browser checks used local in-memory assets and Canvas 3D under harness restrictions. Physical GPU/microphone remain unverified. Its 32 MiB WASM arena plus DOM/graphics is not the 32 KiB neural cache.

In [ ]:
display(HTML(show_table(evidence['recipe']['layers'], ['id', 'in_channels', 'out_channels', 'kernel', 'dilation', 'activation', 'parameters', 'history_frames'])))
print('ONNX SHA-256:', evidence['recipe']['source_onnx_sha256'])
print(evidence['research_choice'])

## 3. Dataset inventory, validation gaps and sample payload

Recorded hashes and frame counts are not fresh reads of absent WAVs. Synthetic voice variants are not independent human speakers; repeated wording is not unseen-language generalization. V1 lacks recorded PCM hashes and its previously found byte-identical controls remain a limitation. All 48 extra V3 test IDs are accounted for by the supplemental OOV manifest. Inspected tests are regression sets, not sealed selection sets. Missing original human-test verbatims cannot be reconstructed from aggregate scores.

In [ ]:
display(HTML(show_table(evidence['audit']['manifests'], ['path', 'rows', 'split_counts', 'recording_ids_shared_across_splits', 'recorded_hashes_shared_across_splits', 'rows_without_recorded_pcm_hash', 'identical_texts_across_splits'])))
print(json.dumps(evidence['audit']['supplemental_v3_test'], indent=2))
display(HTML(show_table(evidence['manifest_audio_audit'], ['path', 'rows', 'missing_audio_at_manifest_paths', 'split_counts'])))
print('Sample recorded training payload:')
print(json.dumps(evidence['sample_manifest'], indent=2))

### Service vocabulary and typed-result examples

```json
{
  "home_entity": {"entity_id": "light.lamp_two", "name": "Lamp Two", "aliases": ["lamp two"], "domain": "light", "exposed": true},
  "music_track": {"id": "dummy_track_001", "name": "Quiet River", "artists": [{"id": "dummy_artist_006", "name": "Luna Rose"}]},
  "output": {"candidate": {"intent": "SET_BRIGHTNESS", "entity_id": "light.lamp_two", "value": 73, "unit": "percent"}, "status": "needs_clarification", "reason": "uncalibrated", "action_executed": false}
}
```

BY is reserved in `<song> [by <artist>]`. Only actual dummy track–artist relationships are compiled. Duplicate titles retain ties. Device-name numbers are not percentages. Catalog changes invalidate stale acceptance evidence. No live service credentials or actions are used.

## 4. All ten application use cases: verbatim / expected / actual

The 44 synthetic recordings are scope diagnostics, not fitting or selection data. Unsupported expected outputs describe requested behavior, not implemented intents. Result: **10/40 raw positive matches**, plus four negative/ambiguous inputs; all final requests refused. This is not comparable to the narrower V6 645/706 result.

The original ten application use cases and the legacy model's ten labels are different. UNKNOWN is a rejection class. The second table explicitly crosswalks every legacy label to a command and the current research output; **it does not claim to run the unavailable legacy checkpoint**.

In [ ]:
display(HTML(show_table(evidence['scope']['rows'], ['use_case', 'family', 'voice', 'text', 'target', 'candidate', 'status', 'supported_by_grammar'])))

In [ ]:
# Crosswalk only: the unavailable legacy checkpoint is not reexecuted.
legacy_examples = {
    'UNKNOWN': 'do not set volume to fifty',
    'POWER_ON': 'turn on bedroom lamp',
    'POWER_OFF': 'turn off kitchen lamp',
    'BRIGHTNESS': 'set brightness of lamp two to seventy three percent',
    'COLOR_SET': 'set bedroom lamp to red',
    'MEDIA_PLAY': 'play quiet river by luna rose',
    'MEDIA_RESUME': 'resume the music',
    'TIME_QUERY': 'what time is it',
    'DATE_QUERY': 'what is the date today',
    'VOLUME': 'set volume to forty two percent',
}
legacy_rows = []
for label, text in legacy_examples.items():
    for row in evidence['scope']['rows']:
        if row['text'] == text:
            legacy_rows.append({
                'legacy_label': label, 'voice': row['voice'], 'verbatim': text,
                'expected_typed_result': row['target'],
                'actual_research_output': row['candidate'],
                'final_decision': row['status'], 'legacy_model_reexecuted': False,
            })
display(HTML(show_table(legacy_rows, [
    'legacy_label', 'voice', 'verbatim', 'expected_typed_result',
    'actual_research_output', 'final_decision', 'legacy_model_reexecuted'
])))

## 5. Selection and qualification

Select within matching development/catalog/frontend protocols. Keep the epoch-zero parent rather than promote worse adaptation. No fitting on this regression ledger. Release needs human/Pi evidence, complete scope, catalog-specific calibration, accepted exact recall ≥90%, observed accepted precision ≥99%, and negative false acceptance ≤1%. Human qualification additionally needs confidence intervals and speaker-cluster analysis; point thresholds are not guarantees.

In [ ]:
candidates = [{'candidate': 'parent phone_s53 / epoch zero', 'dev_phone_error': evidence['selection']['development']['phone_error_rate']}]
candidates += [{'candidate': 'adapted epoch ' + str(r['epoch']), 'dev_phone_error': r['phone_error_rate']} for r in evidence['training']]
display(HTML(show_table(candidates, ['candidate', 'dev_phone_error'])))
print(json.dumps(evidence['release'], indent=2))
print(json.dumps(evidence['duplicates'], indent=2))
display(HTML(show_table([{'trial': k, **v} for k, v in evidence['summaries'].items()], ['trial', 'recordings', 'valid', 'negatives', 'raw_correct', 'accepted_correct', 'wrong_accepts', 'accepted_precision'])))

## 6. Every incorrect and correct command

No `head()` truncation: all rows are in collapsed per-study tables. Train/dev/calibration rows, failed variants and repeated recordings are labeled. Do not pool these into one accuracy statistic. A wrong raw negative candidate is not a false acceptance unless its policy accepted it. V4/V5 accepted flags are diagnostic, never external actions.

In [ ]:
show_complete('Incorrect raw characterization/parsing', [r for r in rows if r['expected'] is not None and not r['raw_exact']])

In [ ]:
show_complete('Wrong diagnostic acceptances', [r for r in rows if r['accepted'] and not r['raw_exact']])

In [ ]:
show_complete('All correct raw outputs, including rejected correct candidates', [r for r in rows if r['raw_exact']])

In [ ]:
show_complete('All negatives and ambiguity cases', [r for r in rows if r['expected'] is None])

## 7. Run the suite and visualization

```bash
python final_suite/run_all.py --browser
python final_suite/run_all.py --qualify
python -m http.server 8000 --bind 127.0.0.1
```

Qualification deliberately fails while evidence is missing. Open `http://localhost:8000/visualizer/` after restoring the payload. Run/step computes real tensors. Select a layer/channel/frame, inspect weighted contributions, download tensors and search every command result. Speed 0.25–3× changes scheduling, not audio. Mic mode is 1× and permission-gated; physical behavior is unverified.

The viewer is not part of Pi inference. Remaining work is reviewed human speech, sealed calibration/selection, missing application families, sustained Pi4 capture/latency/RSS/temperature tests. Passing software must never silently activate actions.